In [1]:
import numpy as np
import sys
import time
import pandas as pd


from lib.envs.gridworld import GridworldEnv

env = GridworldEnv()


## 策略评估方法
def policy_eval(policy, env, discount_factor =1, threshold=0.00001):
	# 初始化各状态的状态值函数
	V = np.zeros(env.nS)
	i = 0
	V_current = np.zeros(env.nS)

	while True:
		value_delta = 0
		# 遍历各状态
		for s in range(env.nS):
			v = 0

			for a, action_prob in enumerate(policy[s]):
				for prob, next_state, reward, done in env.P[s][a]:
					v += action_prob * prob * (reward + discount_factor * V[next_state])
					# v += action_prob * (reward + discount_factor * prob * V_current[next_state]) # 此表达式在程序写作中不恰当
					
			value_delta = max(value_delta, np.abs(v - V_current[s]))
			V[s] = v

		# value_delta = max(value_delta, np.abs(V - V_current))
		i += 1
		V_current = np.copy(V)
		if i <= 4:
			print(i)
			print()
			print(V.reshape(5,5))
		if value_delta < threshold:
			break
	return np.array(V)


In [2]:
## 策略改进方法
v_num = 1
i_num = 1

# 根据传入的四个行为选择值函数最大的索引，返回的是一个索引数组和一个行为策略
def get_max_index(action_values):
	indexs = []
	policy_arr = np.zeros(len(action_values))
	action_max_value = np.max(action_values)

	for i in range(len(action_values)):
		action_value = action_values[i]

		if action_value == action_max_value:
			indexs.append(i)
			policy_arr[i] = 1
	return indexs, policy_arr

# 将策略中每行可能行为改为元组形式，方便对多个方向表示
def change_policy(policys):
	action_tuple = []

	for policy in policys:
		indexs, policy_arr = get_max_index(policy)
		action_tuple.append(tuple(indexs))
	return action_tuple	
	
# policy_improvement是策略改进方法，输入为环境env，策略评估函数policy_eval
# 输出为最优值函数和最优策略
def policy_improvement(env, policy_eval_fn = policy_eval, discount_factor=1.0):
	start_time = time.perf_counter()

	# 初始化一个随机策略
	policy = np.ones([env.nS, env.nA])/env.nA

	while True:
		global i_num
		global v_num

		v_num = 1

		# 评估当前的策略，输出为各状态当前的状态值函数
		V = policy_eval_fn(policy, env, discount_factor)
		#定义一个当前策略是否改变的标识
		policy_stable = True

		#遍历各状态
		for s in range(env.nS):
			# 取出当前状态下最优行为的索引值
			chosen_a = np.argmax(policy[s])
			
			# 初始化行为数组[0,0,0,0]
			action_values = np.zeros(env.nA)
			for a in range(env.nA):
				for prob, next_state, reward, done in env.P[s][a]:
					action_values[a] += prob * (reward + discount_factor * V[next_state])
			
			best_a_arr, policy_arr = get_max_index(action_values)
			if chosen_a not in best_a_arr:
				policy_stable = False
			policy[s] = policy_arr

		i_num += 1

		# 如果当前策略没有发生改变，即已经到了最优策略，返回
		if policy_stable:
			stop_time = time.perf_counter()
			print('Duration time is %.3f seconds'%(stop_time - start_time))
			
			return policy, V
		
env = GridworldEnv()
policy, v = policy_improvement(env)
update_policy_type = change_policy(policy)	

1

[[-1.         -1.25       -1.3125     -1.078125   -1.26953125]
 [-1.25       -1.625      -1.484375    0.         -1.06738281]
 [-1.3125     -1.734375   -1.8046875  -1.20117188 -1.56713867]
 [-1.328125   -1.765625   -1.89257812 -1.7734375  -1.83514404]
 [-1.33203125 -1.77441406 -1.91674805 -1.92254639 -1.93942261]]
2

[[-2.125      -2.578125   -2.61328125 -1.99023438 -2.39916992]
 [-2.578125   -3.09375    -2.62792969  0.         -2.00842285]
 [-2.73828125 -3.35058594 -3.26806641 -2.40216064 -2.95321655]
 [-2.79101562 -3.45214844 -3.6026001  -3.44061279 -3.542099  ]
 [-2.80737305 -3.4876709  -3.73239136 -3.75874329 -3.79492188]]
3

[[-3.3515625  -3.90917969 -3.78515625 -2.79364014 -3.40010071]
 [-3.94042969 -4.45703125 -3.62756348  0.         -2.84043503]
 [-4.20507812 -4.84558105 -4.61947632 -3.50332642 -4.20976925]
 [-4.31390381 -5.06243896 -5.21372986 -5.00447464 -5.13781619]
 [-4.3540802  -5.15914536 -5.46600246 -5.50603557 -5.55842388]]
4

[[-4.63818359 -5.1973877  -4.85093689 -3

In [3]:
print("Policy Probability Distribution:")
policy = pd.DataFrame(policy,columns=['up','right','down','left'])
print(policy)



Policy Probability Distribution:
     up  right  down  left
0   0.0    1.0   1.0   0.0
1   0.0    1.0   1.0   0.0
2   0.0    1.0   1.0   0.0
3   0.0    0.0   1.0   0.0
4   0.0    0.0   1.0   1.0
5   0.0    1.0   0.0   0.0
6   0.0    1.0   0.0   0.0
7   0.0    1.0   0.0   0.0
8   1.0    1.0   1.0   1.0
9   0.0    0.0   0.0   1.0
10  1.0    1.0   0.0   0.0
11  1.0    1.0   0.0   0.0
12  1.0    1.0   0.0   0.0
13  1.0    0.0   0.0   0.0
14  1.0    0.0   0.0   1.0
15  1.0    1.0   0.0   0.0
16  1.0    1.0   0.0   0.0
17  1.0    1.0   0.0   0.0
18  1.0    0.0   0.0   0.0
19  1.0    0.0   0.0   1.0
20  1.0    1.0   0.0   0.0
21  1.0    1.0   0.0   0.0
22  1.0    1.0   0.0   0.0
23  1.0    0.0   0.0   0.0
24  1.0    0.0   0.0   1.0


In [4]:
print("Reshaped Grid Policy (0=up, 1=right, 2=down, 3=left):")
print(np.reshape(np.argmax(policy, axis=1), env.shape))



Reshaped Grid Policy (0=up, 1=right, 2=down, 3=left):
[[1 1 1 2 2]
 [1 1 1 0 3]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]


In [5]:
print("Value Function:")
print(v)



Value Function:
[-3. -2. -1.  0. -1. -2. -1.  0.  0.  0. -3. -2. -1.  0. -1. -4. -3. -2.
 -1. -2. -5. -4. -3. -2. -3.]


In [6]:
print("Reshaped Grid Value Function:")
print(v.reshape(env.shape))

Reshaped Grid Value Function:
[[-3. -2. -1.  0. -1.]
 [-2. -1.  0.  0.  0.]
 [-3. -2. -1.  0. -1.]
 [-4. -3. -2. -1. -2.]
 [-5. -4. -3. -2. -3.]]
